In [1]:
#Complete Python Program
import os
import sqlite3

# ----------------------------------------
# Step 1 : Create Database
# ----------------------------------------
db_name = "retention.db"

# Delete old database (optional)
if os.path.exists(db_name):
    os.remove(db_name)

# Create new database
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# ----------------------------------------
# Step 2 : Create Orders Table
# ----------------------------------------
cursor.execute("""
CREATE TABLE orders(
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT
)
""")

# ----------------------------------------
# Step 3 : Insert Sample Data
# ----------------------------------------
orders = [
(101,1,'2026-08-01'),
(102,1,'2026-08-05'),
(103,2,'2026-08-02'),
(104,2,'2026-08-15'),
(105,3,'2026-08-03'),
(106,3,'2026-08-08'),
(107,4,'2026-08-04'),
(108,4,'2026-08-20')
]

cursor.executemany(
"INSERT INTO orders VALUES (?,?,?)",
orders)

conn.commit()

# ----------------------------------------
# Step 4 : Display Orders
# ----------------------------------------
print("="*70)
print("ORDERS TABLE")
print("="*70)

for row in cursor.execute(
"SELECT * FROM orders ORDER BY customer_id, order_date"):
    print(row)

# ----------------------------------------
# Step 5 : Find 7-Day Repeat Purchases
# ----------------------------------------
query = """
WITH first_order AS
(
SELECT
customer_id,
MIN(order_date) AS first_date
FROM orders
GROUP BY customer_id
)

SELECT DISTINCT
f.customer_id

FROM first_order f
JOIN orders o
ON o.customer_id = f.customer_id

WHERE o.order_date > f.first_date
AND julianday(o.order_date) - julianday(f.first_date) <= 7

ORDER BY f.customer_id;
"""

print("\n")
print("="*70)
print("CUSTOMERS WITH REPEAT PURCHASE WITHIN 7 DAYS")
print("="*70)

for row in cursor.execute(query):
    print(row)

conn.close()



ORDERS TABLE
(101, 1, '2026-08-01')
(102, 1, '2026-08-05')
(103, 2, '2026-08-02')
(104, 2, '2026-08-15')
(105, 3, '2026-08-03')
(106, 3, '2026-08-08')
(107, 4, '2026-08-04')
(108, 4, '2026-08-20')


CUSTOMERS WITH REPEAT PURCHASE WITHIN 7 DAYS
(1,)
(3,)
